In [1]:
import json
import openai

In [2]:
with open("../data/courses.json", "r") as f:
    courses = json.load(f)
with open("../data/glossary.json", "r") as f:
    glossary = json.load(f)

In [3]:
gcodes = glossary.keys()
lcodes = [c["split_course_code"][0] for c in courses]
ccodes = {c["split_course_code"][0] for c in courses}

In [ ]:
diff = ccodes - gcodes
print(len(diff))
buffer = []
for s in diff:
    buffer.append((s, lcodes.count(s)))
buffer.sort(key=lambda x: x[1], reverse=True)
print(buffer)

80
[('RSM', 98), ('HMB', 63), ('VIC', 56), ('TRN', 46), ('ESS', 37), ('BMS', 35), ('DRM', 35), ('MUN', 35), ('NML', 33), ('INS', 30), ('CDN', 26), ('CRE', 24), ('PHS', 24), ('MGY', 24), ('CHC', 23), ('CSE', 22), ('AFR', 22), ('ACT', 21), ('CLT', 20), ('FSL', 18), ('NEW', 18), ('COG', 16), ('EDS', 16), ('DTS', 16), ('CAR', 15), ('LCT', 15), ('URB', 15), ('CJS', 15), ('PHC', 14), ('SMC', 14), ('REN', 12), ('APM', 12), ('UNI', 11), ('PRT', 11), ('INT', 9), ('WRR', 9), ('ARH', 9), ('BIO', 9), ('EST', 9), ('MCS', 9), ('DHU', 9), ('FIN', 8), ('PDC', 8), ('IRW', 8), ('ABP', 8), ('CAS', 8), ('BPM', 7), ('FCS', 7), ('JEG', 7), ('ANA', 6), ('AMS', 6), ('MGR', 6), ('LAS', 6), ('INI', 5), ('EUR', 5), ('WDW', 4), ('BCB', 4), ('JGU', 4), ('STS', 4), ('ENT', 3), ('JSN', 3), ('JLS', 3), ('JSU', 2), ('JFL', 2), ('CTA', 2), ('JIG', 2), ('IFP', 2), ('MIJ', 1), ('JCR', 1), ('JHU', 1), ('MHB', 1), ('JGA', 1), ('JFP', 1), ('JWB', 1), ('JSM', 1), ('JQR', 1), ('ETH', 1), ('CJH', 1), ('EHJ', 1), ('JWE', 1)]


In [38]:
def get_examples(prefix, limit=20):
    examples = []
    for c in courses:
        if c["split_course_code"][0] == prefix:
            examples.append(f"{c["course_code"]} {c["title"]} DESCRIPTION {c["description"]}")
        if len(examples) > 20:
            break
    return examples

def stringify_examples(examples):
    return "\n".join(examples)

In [39]:
client = openai.OpenAI(base_url="http://localhost:1337/v1", api_key="helloworld")

In [40]:
task = """
PREFIX {prefix}

EXAMPLES
{courses}
"""

In [41]:
instructions = """
You are a tool part of a processing pipeline for University of Toronto courses.
Your task is to guess the meaning of 3 letter course codes. 
They are often an acronym or the starting letters of their department / name.
Note that J often indicates a collaboration between two departments.
A list of example courses is provided.
Output just your best guess only, with no extra text or explanation.
"""

In [48]:
convo_template = [
        {"role": "system", "content": instructions},
]
with open("../docs/coursecode_glossary_appendix.json", "r") as f:
    data = json.load(f)
known_codes = data["known"]
limit = 5
for prefix, meaning in known_codes.items():
    convo_template.append({"role": "user", "content":f"PREFIX {prefix}"})
    convo_template.append({"role": "assistant", "content": meaning})
print(convo_template)

[{'role': 'system', 'content': '\nYou are a tool part of a processing pipeline for University of Toronto courses.\nYour task is to guess the meaning of 3 letter course codes. \nThey are often an acronym or the starting letters of their department / name.\nNote that J often indicates a collaboration between two departments.\nA list of example courses is provided.\nOutput just your best guess only, with no extra text or explanation.\n'}, {'role': 'user', 'content': 'PREFIX RSM'}, {'role': 'assistant', 'content': 'Management (Rotman)'}, {'role': 'user', 'content': 'PREFIX HMB'}, {'role': 'assistant', 'content': 'Human Biology'}, {'role': 'user', 'content': 'PREFIX VIC'}, {'role': 'assistant', 'content': 'Victoria College'}, {'role': 'user', 'content': 'PREFIX TRN'}, {'role': 'assistant', 'content': 'Trinity College'}, {'role': 'user', 'content': 'PREFIX ESS'}, {'role': 'assistant', 'content': 'Earth System Science'}, {'role': 'user', 'content': 'PREFIX MUN'}, {'role': 'assistant', 'conten

In [43]:
def send_loop(prefix):
    m = task.format(prefix=prefix, courses=stringify_examples(get_examples(prefix)))
    conversation = convo_template[:] + [{"role": "user", "content": m}]
    response = client.chat.completions.create(
        model="Jan-v3.5-4B-Q4_K_XL",
        messages=conversation,  # conversation history
        max_tokens=20,  # cap response length
        temperature=0.3,  # 0 = deterministic, 2 = very random
        stream=False,  # True to receive tokens as they generate
    )
    guess = response.choices[0].message.content
    return guess

In [ ]:
outputs = dict()
for i, (prefix, c) in enumerate(buffer):
    guess = send_loop(prefix)
    outputs[prefix] = guess
    print(prefix, c, guess)

CHC 23 Christianity and Culture
CAR 15 Caribbean Studies
ENT 3 Entrepreneurship
JSU 2 Jewish Studies
MIJ 1 Molecular Immunology and Genetics
INT 9 Arts & Science Internship Program
BMS 35 Book Media Studies
BPM 7 Buddhism and Psychology
COG 16 Cognitive Science
PHC 14 Pharmaceutical Chemistry
INS 30 Indigenous Studies
FSL 18 French as a Second Language
WDW 4 War, Defence, and World
WRR 9 Writing & Rhetoric
FIN 8 Finnish
ACT 21 Actuarial Science
JCR 1 Jewish Studies
PDC 8 Arts & Science Internship Program
CRE 24 Creativity
BCB 4 Bioinformatics and Computational Biology
JGU 4 Jewish Studies
JHU 1 Joint (Humanities & Social Sciences)
ANA 6 Anatomy
INI 5 Institute of Urban Studies
ARH 9 Archaeology
CSE 22 Critical Equity and Solidarity Studies
JSN 3 Jewish Studies
AMS 6 American Studies
MGR 6 Modern Greek
UNI 11 University of Toronto (University College)
MHB 1 Modern Hebrew
BIO 9 Biology
LCT 15 Literature and Critical Theory
ESS 37 Earth System Science
PHS 24 Public Health
JGA 1 Joint (Geo

In [ ]:
with open("../docs/coursecode_glossary_appendix.json", "r") as f:
    data = json.dump(f)
data["llm_guesses"] = outputs
with open("../docs/coursecode_glossary_appendix.json", "w") as f:
    json.dump(data, f, indent=2)